# Attention U-Net → BUSI — project notebook

**Crash-safe & resumable.** All outputs go to Google Drive, checkpoints are saved
every epoch, and finished work is skipped on re-run. If Colab disconnects, just
**re-run every cell top-to-bottom** — completed seeds are skipped and the
in-progress one resumes from its last epoch (all read back from Drive).

All logic lives in the `busi/` package; this notebook only orchestrates.
Control panel = `busi/config.py`.

## 1. Bootstrap — Colab only

On a fresh Colab runtime, run this to clone the repo + install deps. On a local
kernel (your own `.venv`) **skip this** — just run from the repo root.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Yuval-Naim/DL_attention_busi.git"   # public — no token
REPO_DIR = "DL_attention_busi"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "requirements.txt", "-i", "https://pypi.org/simple"], check=True)

print("cwd:", os.getcwd())

## 2. Mount Drive + set paths (the crash-safety step)

Everything that must survive a disconnect — dataset, checkpoints, results — lives
on **Drive**, never on the temporary Colab disk. Put your BUSI folders
(`benign/ malignant/ normal/`) under `DATA_ROOT`.

In [ ]:
import os
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
else:
    DRIVE = "."                      # local run: keep outputs in the repo

DATA_ROOT = f"{DRIVE}/BUSI"          # your dataset (benign/ malignant/ normal/)
OUT_DIR   = f"{DRIVE}/busi_out"      # checkpoints + results persist here
os.makedirs(f"{OUT_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)
print("DATA_ROOT:", DATA_ROOT, "| exists:", os.path.isdir(DATA_ROOT))
print("OUT_DIR:  ", OUT_DIR)

## 3. Sanity + data check

Expected: **780** images (437/210/133). (If the data isn't on Drive yet, download
it from Kaggle — `aryashah2k/breast-ultrasound-images-dataset` — into `DATA_ROOT`.
Dataset: Al-Dhabyani et al., Data in Brief 2020, CC BY 4.0.)

In [ ]:
import torch, busi
from busi.config import Config
from busi import train as T, experiment as E
from busi.data import list_busi_samples
print("busi", busi.__version__, "| torch", torch.__version__, "| device", T.get_device())

samples = list_busi_samples(DATA_ROOT, classes=("benign", "malignant", "normal"))
print("total samples:", len(samples))          # expect 780
split = E.get_or_make_split(Config(data_root=DATA_ROOT))
print({k: len(v) for k, v in split.items()})

## 4. Configure the run

One config drives everything; outputs point at Drive. `QUICK=True` gives a fast
smoke (1 seed, few epochs).

In [ ]:
QUICK = False
base = Config(
    data_root=DATA_ROOT,
    checkpoints_dir=f"{OUT_DIR}/checkpoints",   # -> Drive (best + resume ckpts)
    results_dir=f"{OUT_DIR}/results",           # -> Drive (per-seed + per-model JSON)
    epochs=(3 if QUICK else 100),
    seeds=([42] if QUICK else [42, 1, 7]),
    ckpt_every=1,                               # save a resume checkpoint every epoch
)
MODELS = ["unet", "attention_unet", "cbam_unet", "scse_unet"]
print("epochs:", base.epochs, "| seeds:", base.seeds, "| out:", base.results_dir)

## 5. Run — one model per cell (resumable)

Each cell trains all seeds for one model. If it crashes, **just re-run that cell**
(or the whole notebook) — finished seeds are skipped, the in-progress seed resumes
from its last epoch. Live per-epoch logs show progress.

In [ ]:
E.run_seeds("unet", cfg=base)               # baseline

In [ ]:
E.run_seeds("attention_unet", cfg=base)      # paper's additive gate

In [ ]:
E.run_seeds("cbam_unet", cfg=base)           # desired

In [ ]:
E.run_seeds("scse_unet", cfg=base)           # stretch

## 6. Results table & figures

Built from whatever finished (works even if some models are still pending).

In [ ]:
results = E.collect_results(MODELS, base.results_dir)
print(E.make_results_table(results))

# Attention-map figures for the attention model (best-seed checkpoint), saved to Drive.
if any(r["model"] == "attention_unet" for r in results):
    cfg_att = E._cfg_for(base, "attention_unet", base.seeds[0])
    ckpt = f"{cfg_att.checkpoints_dir}/{cfg_att.experiment_name}.pt"
    figs = E.save_prediction_figures(cfg_att, ckpt, f"{OUT_DIR}/figures", n=6)
    print("saved figures:", figs)

## 7. Notes

- **Recover from a crash:** re-run all cells. Done seeds skip; the in-progress one
  resumes from Drive — nothing lost beyond the current epoch.
- **Stretch — loss study** (Focal-Tversky on the best variant):
  ```python
  d = {**base.to_dict(), "loss_name": "focal_tversky", "experiment_name": ""}
  E.run_seeds("attention_unet", cfg=Config(**d))
  ```
- Prefer a GPU runtime (Runtime → Change runtime type → **T4 GPU**).